# Extract phases from solutions

Created 05/03/2026

Objectives:
* Evalute the index from [this paper](https://doi.org/10.1063/5.0055996) on the qudit cluster chain. Should be faster than manually evaluating all the time.

# Package imports

In [1]:
import sys

In [2]:
sys.path.append("../../")

In [3]:
import h5py
from tenpy.tools import hdf5_io
import tenpy
import tenpy.linalg.np_conserved as npc

import os
import pickle

In [4]:
import numpy as np
import jax.numpy as jnp

import pandas as pd

import matplotlib.pyplot as plt
import matplotlib

In [70]:
from collections import Counter

# Load data

In [5]:
DATA_DIR = r"../../data/qudit_cluster_chain_d_7_L_50.h5"

In [6]:
with h5py.File(DATA_DIR, 'r') as f:
    psi = hdf5_io.load_from_hdf5(f)

# Check expectations

In [8]:
Z_expectation_values = psi.expectation_value('Z')

In [9]:
Z_expectation_values.shape

(50,)

In [10]:
np.round(Z_expectation_values, 4)

array([-0.901 -0.4339j, -0.    -0.j    , -0.    -0.j    ,  0.    +0.j    ,
       -0.    -0.j    , -0.    +0.j    , -0.    -0.j    , -0.    -0.j    ,
       -0.    -0.j    , -0.    -0.j    , -0.    -0.j    ,  0.    -0.j    ,
        0.    -0.j    , -0.    +0.j    ,  0.    +0.j    , -0.    -0.j    ,
        0.    -0.j    ,  0.    -0.j    ,  0.    -0.j    ,  0.    +0.j    ,
        0.    -0.j    , -0.    -0.j    ,  0.    -0.j    ,  0.    +0.j    ,
       -0.    -0.j    , -0.    +0.j    , -0.    +0.j    ,  0.    +0.j    ,
        0.    -0.j    ,  0.    +0.j    ,  0.    +0.j    , -0.    -0.j    ,
       -0.    -0.j    , -0.    -0.j    , -0.    +0.j    , -0.    -0.j    ,
        0.    +0.j    ,  0.    -0.j    , -0.    -0.j    ,  0.    +0.j    ,
        0.    +0.j    ,  0.    -0.j    , -0.    +0.j    , -0.    -0.j    ,
       -0.    -0.j    , -0.    -0.j    , -0.    +0.j    , -0.    +0.j    ,
        0.    +0.j    , -0.3408-0.1537j])

All vanishing, as expected.

In [12]:
X_expectation_values = psi.expectation_value('X')

In [13]:
X_expectation_values.shape

(50,)

In [14]:
np.round(X_expectation_values, 4)

array([ 0.,  0., -0., -0., -0., -0.,  0.,  0., -0.,  0., -0.,  0., -0.,
       -0.,  0., -0., -0., -0.,  0.,  0., -0., -0.,  0., -0., -0., -0.,
       -0.,  0., -0., -0., -0.,  0., -0., -0.,  0.,  0., -0., -0.,  0.,
        0., -0., -0., -0.,  0.,  0.,  0., -0.,  0.,  0., -0.])

In [15]:
psi.L

50

# Calculate Kapustin indices

In [19]:
XI_sym = [('X', i) for i in range(14, 34, 2)]
IX_sym = [('X', i) for i in range(15, 34, 2)]
XX_sym = [('X', i) for i in range(14, 34)]

XI_sym_hc = [('Xhc', i) for i in range(14, 34, 2)]
IX_sym_hc = [('Xhc', i) for i in range(15, 34, 2)]
XX_sym_hc = [('Xhc', i) for i in range(14, 34)]

In [18]:
XI_b = [('Zhc', 33),]
IX_b = [('Z', 34),]
XX_b = [('Zhc', 33), ('Z', 34)]

XI_b_hc = [('Z', 33),]
IX_b_hc = [('Zhc', 34),]
XX_b_hc = [('Z', 33), ('Zhc', 34)]

In [20]:
test_terms = [
    *XX_b_hc,
    *IX_sym,
    *XI_b,
    *IX_sym_hc,
    *IX_b
]

In [26]:
exp = psi.expectation_value_term(test_terms)

In [24]:
def expectation_to_discrete_phase(expectation, d=7):
    phase = np.imag(np.log(expectation))
    discrete_phase = phase/(2*np.pi/d)

    return discrete_phase

In [27]:
expectation_to_discrete_phase(exp)

-1.0000000000000002

In [37]:
stabilizers = [
    [('Zhc', 31), ('X', 32), ('Z', 33)],
    [('Z', 32), ('X', 33), ('Zhc', 34)],
    [('Zhc', 33), ('X', 34), ('Z', 35)],
    [('Z', 34), ('X', 35), ('Zhc', 36)],
    # And conjugate terms
    [('Z', 31), ('Xhc', 32), ('Zhc', 33)],
    [('Zhc', 32), ('Xhc', 33), ('Z', 34)],
    [('Z', 33), ('Xhc', 34), ('Zhc', 35)],
    [('Zhc', 34), ('Xhc', 35), ('Z', 36)]
]

In [38]:
stabilizer_indicators = np.random.binomial(1, 0.5, (3, len(stabilizers)))

In [39]:
stabilizer_indicators

array([[0, 1, 1, 0, 0, 1, 0, 1],
       [1, 1, 1, 1, 1, 0, 0, 0],
       [1, 0, 1, 0, 1, 0, 0, 0]])

In [40]:
random_stablizers = [
    [s for (s, i) in zip(stabilizers, indicators) if (i==1)]
    for indicators in stabilizer_indicators
]

In [41]:
random_stablizers

[[[('Z', 32), ('X', 33), ('Zhc', 34)],
  [('Zhc', 33), ('X', 34), ('Z', 35)],
  [('Zhc', 32), ('Xhc', 33), ('Z', 34)],
  [('Zhc', 34), ('Xhc', 35), ('Z', 36)]],
 [[('Zhc', 31), ('X', 32), ('Z', 33)],
  [('Z', 32), ('X', 33), ('Zhc', 34)],
  [('Zhc', 33), ('X', 34), ('Z', 35)],
  [('Z', 34), ('X', 35), ('Zhc', 36)],
  [('Z', 31), ('Xhc', 32), ('Zhc', 33)]],
 [[('Zhc', 31), ('X', 32), ('Z', 33)],
  [('Zhc', 33), ('X', 34), ('Z', 35)],
  [('Z', 31), ('Xhc', 32), ('Zhc', 33)]]]

In [47]:
list(random_stablizers[0])

[[('Z', 32), ('X', 33), ('Zhc', 34)],
 [('Zhc', 33), ('X', 34), ('Z', 35)],
 [('Zhc', 32), ('Xhc', 33), ('Z', 34)],
 [('Zhc', 34), ('Xhc', 35), ('Z', 36)]]

In [48]:
def flatten_list(l):
    return [x for l1 in l for x in l1]

In [49]:
flatten_list(random_stablizers[0])

[('Z', 32),
 ('X', 33),
 ('Zhc', 34),
 ('Zhc', 33),
 ('X', 34),
 ('Z', 35),
 ('Zhc', 32),
 ('Xhc', 33),
 ('Z', 34),
 ('Zhc', 34),
 ('Xhc', 35),
 ('Z', 36)]

In [52]:
test_terms = [
    *flatten_list(random_stablizers[0]),
    *XX_b_hc,
    *IX_sym,
    *XI_b,
    *flatten_list(random_stablizers[1]),
    *IX_sym_hc,
    *IX_b,
    *flatten_list(random_stablizers[2]),
]

In [53]:
expectation_to_discrete_phase(psi.expectation_value_term(test_terms))

-1.0

In [62]:
random_stablizers = [
    [s for (s, i) in zip(stabilizers, indicators) if (i==1)]
    for indicators in stabilizer_indicators
]

In [63]:
test_terms = [
    *XX_b_hc,
    *flatten_list(random_stablizers[0]),
    *IX_sym,
    *flatten_list(random_stablizers[1]),
    *XI_b,
    *IX_sym_hc,
    *flatten_list(random_stablizers[2]),
    *IX_b,
]

In [64]:
expectation_to_discrete_phase(psi.expectation_value_term(test_terms))

0.0

Start writing functions.

In [79]:
def sample():
    stabilizer_indicators = np.random.binomial(1, 0.5, (3, len(stabilizers)))

    random_stablizers = [
        [s for (s, i) in zip(stabilizers, indicators) if (i==1)]
        for indicators in stabilizer_indicators
    ]

    terms = [
        *flatten_list(random_stablizers[0]),
        *XX_b_hc,
        *IX_sym,
        *XI_b,
        *flatten_list(random_stablizers[1]),
        *IX_sym_hc,
        *IX_b,
        *flatten_list(random_stablizers[2]),
    ]

    out = expectation_to_discrete_phase(psi.expectation_value_term(terms))

    return out

In [80]:
test_samples = [sample() for _ in range(100)]

In [81]:
Counter(test_samples)

Counter({-1.0000000000000002: 40,
         -1.0: 30,
         -0.9999999999999999: 26,
         -0.9999999999999998: 2,
         -1.0000000000000004: 2})

In [134]:
def group_addition(x, y, d=7):
    return ((x[0] + y[0])%d, (x[1] + y[1])%d)

In [138]:
def group_symmetry(g, hc):
    if hc:
        return XI_sym_hc*g[0] + IX_sym_hc*g[1]
    else:
        return XI_sym*g[0] + IX_sym*g[1]

In [139]:
def group_boundary_operator(g, hc):
    if hc:
        return XI_b_hc*g[0] + IX_b_hc*g[1]
    else:
        return XI_b*g[0] + IX_b*g[1]

In [77]:
# Booleans indicate hermitian conjugate or not.
sym_dict = {
    ('II', False): [],
    ('II', True): [],
    ('XI', False): [('X', i) for i in range(14, 34, 2)],
    ('XI', True): [('Xhc', i) for i in range(14, 34, 2)],
    ('IX', False): [('X', i) for i in range(15, 34, 2)],
    ('IX', True): [('Xhc', i) for i in range(15, 34, 2)],
    ('XX', False): [('X', i) for i in range(14, 34)],
    ('XX', True): [('Xhc', i) for i in range(14, 34)]
}

In [78]:
# Booleans again indicate hermitian conjugate or not.
b_op_dict = {
    ('II', False): [],
    ('II', True): [],
    ('XI', False):  [('Zhc', 33),],
    ('XI', True): [('Z', 33),],
    ('IX', False): [('Z', 34),],
    ('IX', True): [('Zhc', 34),],
    ('XX', False): [('Zhc', 33), ('Z', 34)],
    ('XX', True): [('Z', 33), ('Zhc', 34)]
}

In [115]:
def raw_index(g, h):    
    stabilizer_indicators = np.random.binomial(1, 0.5, (3, len(stabilizers)))

    random_stablizers = [
        [s for (s, i) in zip(stabilizers, indicators) if (i==1)]
        for indicators in stabilizer_indicators
    ]

    gh = group_addition[(g, h)]

    terms = [
        *flatten_list(random_stablizers[0]),
        *b_op_dict[(gh, True)],
        *sym_dict[(g, False)],
        *b_op_dict[(h, False)],
        *flatten_list(random_stablizers[1]),
        *sym_dict[(g, True)],
        *b_op_dict[(g, False)],
        *flatten_list(random_stablizers[2]),
    ]

    exp = psi.expectation_value_term(terms)

    return exp

In [116]:
def index(g, h):    
    exp = raw_index(g, h)
    out = expectation_to_discrete_phase(exp)

    return out

Debug.

In [110]:
test_samples = [index('IX', 'XI') for _ in range(100)]

In [111]:
Counter(test_samples)

Counter({-1.0000000000000002: 41,
         -1.0: 28,
         -0.9999999999999999: 23,
         -1.0000000000000004: 5,
         -0.9999999999999998: 3})

In [112]:
group_pairs = [
    ('IX', 'XI'),
    ('XI', 'IX'),
    ('IX', 'XX'),
    ('XX', 'IX'),
    ('XX', 'XI'),
    ('XI', 'XX')
]

In [113]:
results = dict()

for g, h in group_pairs:
    print(g,h)
    current_results = Counter([index(g, h) for _ in range(100)])
    results[(g,h)] = current_results

IX XI
XI IX
IX XX


/tmp/ipykernel_6451/1496427488.py:2: RuntimeWarning: invalid value encountered in log
  phase = np.imag(np.log(expectation))


XX IX
XX XI
XI XX


In [114]:
results

{('IX',
  'XI'): Counter({-1.0000000000000002: 48,
          -1.0: 27,
          -0.9999999999999999: 17,
          -1.0000000000000004: 6,
          -0.9999999999999998: 2}),
 ('XI', 'IX'): Counter({0.0: 100}),
 ('IX', 'XX'): Counter({0.0: 100}),
 ('XX', 'IX'): Counter({0.0: 100}),
 ('XX', 'XI'): Counter({0.0: 100}),
 ('XI', 'XX'): Counter({0.0: 100})}

In [117]:
raw_index('XI', 'IX')

array(1.)

In [121]:
test_exps = [raw_index('XI', 'IX') for _ in range(100)]

In [123]:
test_exps = [raw_index('XI', 'XX') for _ in range(100)]

In [125]:
test_exps = [raw_index('XX', 'XI') for _ in range(100)]

In [127]:
g = 'XI'
h = 'XX'

In [128]:
stabilizer_indicators = np.random.binomial(1, 0.5, (3, len(stabilizers)))

random_stablizers = [
    [s for (s, i) in zip(stabilizers, indicators) if (i==1)]
    for indicators in stabilizer_indicators
]

gh = group_addition[(g, h)]

In [129]:
gh

'IX'

In [130]:
random_stabilizers = [[], [], []]

In [131]:
terms = [
    *flatten_list(random_stablizers[0]),
    *b_op_dict[(gh, True)],
    *sym_dict[(g, False)],
    *b_op_dict[(h, False)],
    *flatten_list(random_stablizers[1]),
    *sym_dict[(g, True)],
    *b_op_dict[(g, False)],
    *flatten_list(random_stablizers[2]),
]

In [133]:
b_op_dict[(gh, True)]

[('Zhc', 34)]

In [140]:
def raw_index(g, h):    
    stabilizer_indicators = np.random.binomial(1, 0.5, (3, len(stabilizers)))

    random_stablizers = [
        [s for (s, i) in zip(stabilizers, indicators) if (i==1)]
        for indicators in stabilizer_indicators
    ]

    gh = group_addition(g,h)

    terms = [
        *flatten_list(random_stablizers[0]),
        *group_boundary_operator(gh, True),
        *group_symmetry(g, False),
        *group_boundary_operator(h, False),
        *flatten_list(random_stablizers[1]),
        *group_symmetry(g, True),
        *group_boundary_operator(g, False),
        *flatten_list(random_stablizers[2]),
    ]

    exp = psi.expectation_value_term(terms)

    return exp

In [143]:
def index(g, h):    
    exp = raw_index(g, h)
    out = expectation_to_discrete_phase(exp)

    return out

In [144]:
test_indices = [index((0,1), (1,1)) for _ in range(100)]

Come up with test cases:

In [147]:
d=7

In [148]:
test_params = [
    [((0,1), (i, 0)) for i in range(d)],
    [((i, 0), (0,1)) for i in range(d)],
    [((3, 4), (i, 5)) for i in range(d)],
    [((i, 5), (3, 4)) for i in range(d)]
]

In [151]:
results = list()

for l in test_params:
    print(l)
    results.append(list())

    for x, y in l:
        print((x,y))
        results[-1].append(list())
        for _ in range(30):
            results[-1][-1].append(index(x,y))

out = np.array(results)

[((0, 1), (0, 0)), ((0, 1), (1, 0)), ((0, 1), (2, 0)), ((0, 1), (3, 0)), ((0, 1), (4, 0)), ((0, 1), (5, 0)), ((0, 1), (6, 0))]
((0, 1), (0, 0))
((0, 1), (1, 0))
((0, 1), (2, 0))
((0, 1), (3, 0))
((0, 1), (4, 0))
((0, 1), (5, 0))
((0, 1), (6, 0))
[((0, 0), (0, 1)), ((1, 0), (0, 1)), ((2, 0), (0, 1)), ((3, 0), (0, 1)), ((4, 0), (0, 1)), ((5, 0), (0, 1)), ((6, 0), (0, 1))]
((0, 0), (0, 1))
((1, 0), (0, 1))
((2, 0), (0, 1))
((3, 0), (0, 1))
((4, 0), (0, 1))
((5, 0), (0, 1))
((6, 0), (0, 1))
[((3, 4), (0, 5)), ((3, 4), (1, 5)), ((3, 4), (2, 5)), ((3, 4), (3, 5)), ((3, 4), (4, 5)), ((3, 4), (5, 5)), ((3, 4), (6, 5))]
((3, 4), (0, 5))
((3, 4), (1, 5))
((3, 4), (2, 5))
((3, 4), (3, 5))
((3, 4), (4, 5))
((3, 4), (5, 5))
((3, 4), (6, 5))
[((0, 5), (3, 4)), ((1, 5), (3, 4)), ((2, 5), (3, 4)), ((3, 5), (3, 4)), ((4, 5), (3, 4)), ((5, 5), (3, 4)), ((6, 5), (3, 4))]
((0, 5), (3, 4))
((1, 5), (3, 4))
((2, 5), (3, 4))
((3, 5), (3, 4))
((4, 5), (3, 4))
((5, 5), (3, 4))
((6, 5), (3, 4))


In [153]:
out.shape

(4, 7, 30)

In [154]:
out[0]

array([[ 0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,
         0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,
         0.,  0.,  0.,  0.],
       [-1., -1., -1., -1., -1., -1., -1., -1., -1., -1., -1., -1., -1.,
        -1., -1., -1., -1., -1., -1., -1., -1., -1., -1., -1., -1., -1.,
        -1., -1., -1., -1.],
       [-2., -2., -2., -2., -2., -2., -2., -2., -2., -2., -2., -2., -2.,
        -2., -2., -2., -2., -2., -2., -2., -2., -2., -2., -2., -2., -2.,
        -2., -2., -2., -2.],
       [-3., -3., -3., -3., -3., -3., -3., -3., -3., -3., -3., -3., -3.,
        -3., -3., -3., -3., -3., -3., -3., -3., -3., -3., -3., -3., -3.,
        -3., -3., -3., -3.],
       [ 3.,  3.,  3.,  3.,  3.,  3.,  3.,  3.,  3.,  3.,  3.,  3.,  3.,
         3.,  3.,  3.,  3.,  3.,  3.,  3.,  3.,  3.,  3.,  3.,  3.,  3.,
         3.,  3.,  3.,  3.],
       [ 2.,  2.,  2.,  2.,  2.,  2.,  2.,  2.,  2.,  2.,  2.,  2.,  2.,
         2.,  2.,  2.,  2.,  2.,  2.,  2.,  2.,  2.,

In [155]:
out[1]

array([[0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
       [0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
       [0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
       [0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
       [0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
       [0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
       [0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.]])

In [156]:
out[2]

array([[ 0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,
         0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,
         0.,  0.,  0.,  0.],
       [ 3.,  3.,  3.,  3.,  3.,  3.,  3.,  3.,  3.,  3.,  3.,  3.,  3.,
         3.,  3.,  3.,  3.,  3.,  3.,  3.,  3.,  3.,  3.,  3.,  3.,  3.,
         3.,  3.,  3.,  3.],
       [-1., -1., -1., -1., -1., -1., -1., -1., -1., -1., -1., -1., -1.,
        -1., -1., -1., -1., -1., -1., -1., -1., -1., -1., -1., -1., -1.,
        -1., -1., -1., -1.],
       [ 2.,  2.,  2.,  2.,  2.,  2.,  2.,  2.,  2.,  2.,  2.,  2.,  2.,
         2.,  2.,  2.,  2.,  2.,  2.,  2.,  2.,  2.,  2.,  2.,  2.,  2.,
         2.,  2.,  2.,  2.],
       [-2., -2., -2., -2., -2., -2., -2., -2., -2., -2., -2., -2., -2.,
        -2., -2., -2., -2., -2., -2., -2., -2., -2., -2., -2., -2., -2.,
        -2., -2., -2., -2.],
       [ 1.,  1.,  1.,  1.,  1.,  1.,  1.,  1.,  1.,  1.,  1.,  1.,  1.,
         1.,  1.,  1.,  1.,  1.,  1.,  1.,  1.,  1.,

In [157]:
out[3]

array([[-1., -1., -1., -1., -1., -1., -1., -1., -1., -1., -1., -1., -1.,
        -1., -1., -1., -1., -1., -1., -1., -1., -1., -1., -1., -1., -1.,
        -1., -1., -1., -1.],
       [-1., -1., -1., -1., -1., -1., -1., -1., -1., -1., -1., -1., -1.,
        -1., -1., -1., -1., -1., -1., -1., -1., -1., -1., -1., -1., -1.,
        -1., -1., -1., -1.],
       [-1., -1., -1., -1., -1., -1., -1., -1., -1., -1., -1., -1., -1.,
        -1., -1., -1., -1., -1., -1., -1., -1., -1., -1., -1., -1., -1.,
        -1., -1., -1., -1.],
       [-1., -1., -1., -1., -1., -1., -1., -1., -1., -1., -1., -1., -1.,
        -1., -1., -1., -1., -1., -1., -1., -1., -1., -1., -1., -1., -1.,
        -1., -1., -1., -1.],
       [-1., -1., -1., -1., -1., -1., -1., -1., -1., -1., -1., -1., -1.,
        -1., -1., -1., -1., -1., -1., -1., -1., -1., -1., -1., -1., -1.,
        -1., -1., -1., -1.],
       [-1., -1., -1., -1., -1., -1., -1., -1., -1., -1., -1., -1., -1.,
        -1., -1., -1., -1., -1., -1., -1., -1., -1.,

Now test with the symmetries acting in the entire region.

In [158]:
large_XI_sym = [('X', i) for i in range(14, 44, 2)]
large_IX_sym = [('X', i) for i in range(15, 44, 2)]

large_XI_sym_hc = [('Xhc', i) for i in range(14, 44, 2)]
large_IX_sym_hc = [('Xhc', i) for i in range(15, 44, 2)]

In [159]:
def large_group_symmetry(g, hc):
    if hc:
        return large_XI_sym_hc*g[0] + large_IX_sym_hc*g[1]
    else:
        return large_XI_sym*g[0] + large_IX_sym*g[1]

In [160]:
def raw_index_2(g, h):    
    stabilizer_indicators = np.random.binomial(1, 0.5, (3, len(stabilizers)))

    random_stablizers = [
        [s for (s, i) in zip(stabilizers, indicators) if (i==1)]
        for indicators in stabilizer_indicators
    ]

    gh = group_addition(g,h)

    terms = [
        *flatten_list(random_stablizers[0]),
        *group_boundary_operator(gh, True),
        *large_group_symmetry(g, False),
        *group_boundary_operator(h, False),
        *flatten_list(random_stablizers[1]),
        *large_group_symmetry(g, True),
        *group_boundary_operator(g, False),
        *flatten_list(random_stablizers[2]),
    ]

    exp = psi.expectation_value_term(terms)

    return exp

In [161]:
def index_2(g, h):    
    exp = raw_index_2(g, h)
    out = expectation_to_discrete_phase(exp)

    return out

In [162]:
results = list()

for l in test_params:
    print(l)
    results.append(list())

    for x, y in l:
        print((x,y))
        results[-1].append(list())
        for _ in range(30):
            results[-1][-1].append(index_2(x,y))

out = np.array(results)

[((0, 1), (0, 0)), ((0, 1), (1, 0)), ((0, 1), (2, 0)), ((0, 1), (3, 0)), ((0, 1), (4, 0)), ((0, 1), (5, 0)), ((0, 1), (6, 0))]
((0, 1), (0, 0))
((0, 1), (1, 0))
((0, 1), (2, 0))
((0, 1), (3, 0))
((0, 1), (4, 0))
((0, 1), (5, 0))
((0, 1), (6, 0))
[((0, 0), (0, 1)), ((1, 0), (0, 1)), ((2, 0), (0, 1)), ((3, 0), (0, 1)), ((4, 0), (0, 1)), ((5, 0), (0, 1)), ((6, 0), (0, 1))]
((0, 0), (0, 1))
((1, 0), (0, 1))
((2, 0), (0, 1))
((3, 0), (0, 1))
((4, 0), (0, 1))
((5, 0), (0, 1))
((6, 0), (0, 1))
[((3, 4), (0, 5)), ((3, 4), (1, 5)), ((3, 4), (2, 5)), ((3, 4), (3, 5)), ((3, 4), (4, 5)), ((3, 4), (5, 5)), ((3, 4), (6, 5))]
((3, 4), (0, 5))
((3, 4), (1, 5))
((3, 4), (2, 5))
((3, 4), (3, 5))
((3, 4), (4, 5))
((3, 4), (5, 5))
((3, 4), (6, 5))
[((0, 5), (3, 4)), ((1, 5), (3, 4)), ((2, 5), (3, 4)), ((3, 5), (3, 4)), ((4, 5), (3, 4)), ((5, 5), (3, 4)), ((6, 5), (3, 4))]
((0, 5), (3, 4))
((1, 5), (3, 4))
((2, 5), (3, 4))
((3, 5), (3, 4))
((4, 5), (3, 4))
((5, 5), (3, 4))
((6, 5), (3, 4))


In [163]:
out[0]

array([[-1.,  0.,  1.,  1., -1.,  1.,  0., -1.,  0.,  0.,  0.,  0.,  0.,
         0., -1.,  1.,  0.,  1.,  0.,  1.,  0.,  0.,  0.,  0.,  1.,  0.,
         0.,  0., -1.,  0.],
       [ 0.,  0., -1., -1., -2.,  0., -1.,  0.,  0., -2.,  0.,  0.,  0.,
        -2.,  0., -2., -1., -2., -1., -1.,  0., -1.,  0.,  0.,  0., -1.,
         0., -2., -1., -2.],
       [-1., -1., -2., -1., -2., -3., -3., -2., -2., -3., -2., -2., -2.,
        -1., -2., -1., -1., -1., -2., -1., -2., -3., -3., -2., -3., -3.,
        -3., -2., -2., -2.],
       [-3., -2., -2., -3.,  3., -3.,  3., -3., -3., -2., -2., -2., -3.,
        -3.,  3., -3.,  3.,  3., -3., -3., -2., -3.,  3., -3., -2., -3.,
        -3., -3., -3., -3.],
       [ 3.,  2.,  3.,  2.,  3.,  2.,  2.,  3., -3.,  3.,  3.,  3., -3.,
         3.,  3., -3., -3., -3.,  3.,  3.,  2.,  3.,  2.,  3.,  3.,  2.,
        -3., -3.,  3.,  3.],
       [ 3.,  2.,  3.,  3.,  2.,  1.,  3.,  1.,  1.,  2.,  2.,  3.,  3.,
         2.,  2.,  2.,  2.,  2.,  3.,  2.,  2.,  2.,

In [164]:
out[1]

array([[ 0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,
         0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,
         0.,  0.,  0.,  0.],
       [ 1.,  2.,  1.,  2.,  2.,  0.,  1.,  1.,  1.,  0.,  2.,  1.,  0.,
         2.,  1.,  2.,  2.,  1.,  1.,  1.,  2.,  0.,  0.,  2.,  1.,  2.,
         0.,  1.,  1.,  2.],
       [ 2.,  2.,  0.,  0.,  2., -3.,  2.,  2.,  2.,  2., -3.,  0.,  2.,
         2.,  2.,  0.,  0.,  0.,  2.,  0.,  0.,  0.,  2.,  2., -3., -3.,
         2.,  2.,  2.,  2.],
       [ 0.,  3.,  0., -1.,  3.,  0.,  3.,  3.,  3.,  3.,  3.,  0.,  0.,
        -1.,  3.,  3., -1.,  3.,  3.,  3.,  0.,  0.,  3.,  3.,  3.,  0.,
         3., -1.,  3.,  3.],
       [-3.,  1.,  0.,  0., -3., -3., -3., -3.,  1., -3., -3.,  0.,  0.,
        -3., -3., -3., -3., -3.,  0.,  0.,  0., -3., -3.,  1., -3.,  0.,
        -3.,  0., -3.,  1.],
       [-2., -2., -2.,  0.,  0.,  3.,  3.,  0.,  0.,  3.,  0.,  0.,  3.,
         3.,  3.,  3., -2., -2.,  3., -2.,  0.,  0.,

In [165]:
out[2]

array([[-3.,  1., -2., -2.,  1.,  0.,  1., -3., -3., -2., -3., -3.,  1.,
        -2.,  1., -3.,  1., -3., -3., -3.,  0., -2.,  1.,  1., -2., -3.,
         0.,  1.,  0., -3.],
       [ 0.,  3., -3.,  1.,  1., -3.,  3.,  0.,  0.,  0., -3.,  0.,  0.,
         0.,  1.,  1.,  0.,  0.,  0., -3., -3.,  0.,  0.,  0.,  0.,  3.,
         1.,  1.,  3., -3.],
       [ 3.,  3.,  1.,  3.,  0.,  3.,  0., -3.,  1., -3.,  0.,  3.,  3.,
         0., -3.,  0.,  0.,  0.,  0.,  3., -3.,  3.,  3.,  1., -1.,  0.,
        -1., -3., -3., -3.],
       [ 0., -1.,  3.,  3.,  3.,  0.,  0.,  0.,  3.,  0., -1., -1., -1.,
        -1., -1., -3., -1.,  3., -1.,  3.,  2.,  0., -1., -1.,  2.,  3.,
         0.,  0.,  3.,  0.],
       [ 3., -1.,  3., -1.,  3., -2.,  3.,  3.,  3.,  2.,  0.,  3.,  3.,
        -1., -1., -1., -1., -1., -1.,  3.,  2., -1., -1., -1., -2., -1.,
         2., -1., -1.,  2.],
       [ 3., -1., -2., -1.,  3., -1., -1., -2., -2.,  2.,  2.,  2.,  1.,
        -1., -2.,  2.,  2., -1.,  2.,  2., -1., -2.,

In [166]:
out[3]

array([[-1., -1., -3.,  1., -1., -1., -3., -1., -3., -1., -1., -1., -1.,
         1., -1., -1., -1.,  1., -1., -3.,  1., -1.,  1., -3., -1., -1.,
        -3., -3., -1., -1.],
       [ 2.,  3.,  3.,  2.,  3., -2.,  0., -3., -3., -3.,  1.,  3.,  3.,
         3.,  3., -3.,  3., -3., -1., -3.,  0.,  1., -2., -1., -1.,  3.,
        -3.,  3.,  3., -2.],
       [-2.,  0.,  0.,  0., -3.,  2.,  0.,  3.,  0.,  0., -2.,  0., -3.,
        -2.,  0., -2.,  0.,  0., -3.,  0.,  0., -2.,  2.,  0.,  0.,  2.,
         0.,  2.,  0.,  3.],
       [ 1.,  2., -1.,  1.,  2., -3.,  1.,  0.,  1., -1.,  2.,  2., -3.,
         1., -1.,  2.,  0.,  2.,  3., -3., -1.,  2., -3.,  2., -1.,  2.,
         0., -3., -1.,  2.],
       [ 0., -2.,  3., -1., -2.,  1.,  1.,  3., -3.,  1., -3.,  3.,  1.,
         3.,  3., -1.,  1.,  3., -3., -1., -3., -3.,  1., -1.,  3., -1.,
         3.,  0., -1., -3.],
       [ 0., -2., -2.,  3.,  3., -2.,  1.,  0.,  2.,  3.,  0.,  3., -2.,
        -2.,  0.,  0.,  0.,  3., -2., -2.,  0.,  1.,